# 4. RQ5: Capacidades sensíveis à segurança e sua deriva nas cópias

**Pergunta:** as skills mandam o agente fazer coisas perigosas? E, quando uma skill é copiada e editada, a cópia fica mais perigosa?

**Por que importa:** o agente executa o que a skill manda, com as permissões do usuário. Uma cópia editada que acrescenta `curl … | sh` ou `--dangerously-skip-permissions` é o equivalente, sem registro e sem revisão, a um pacote adulterado na cadeia de suprimentos.

**Como medimos:** 9 categorias de capacidade detectadas por expressões regulares no texto, cada uma com nível de gravidade e a CWE mais próxima. (1) prevalência; (2) deriva: o que cada variante acrescenta ou remove em relação à versão mais copiada da família, com teste de McNemar; (3) regressão logística para ver que tipo de variante acrescenta capacidades.

**Importante:** medimos **capacidade** (potencial de dano), não vulnerabilidade nem ataque confirmado.

---

### English summary

**What we measure.** Not vulnerabilities and not attacks, but *security-sensitive capabilities*: instructions that give the agent the power to cause harm if the skill were malicious, tampered with in a copy, or misused. Each capability maps to the closest CWE weakness class and a severity tier.

**Methodology step 3: quantitative + statistical modelling.**

* RQ5.1 Prevalence of each capability, also weighted by copies (exposure).
* RQ5.2 Drift: within each variant family, which capabilities an edited copy **adds** or **removes** relative to the most-copied member. Additions vs removals are tested with an exact McNemar test.
* RQ5.3 Logistic regression: which variants add capabilities?

**Methodology step 4 (validation)** happens in notebook 05: manual labelling to measure detector precision.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as smf

In [ ]:
skills = clean.strict(clean.load_skills())
risk_cols = [c for c in skills.columns if c.startswith("risk_")]
pd.DataFrame({"capability": [c[5:] for c in risk_cols],
              "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
              "closest CWE": [patterns.RISK_CWE[c[5:]] for c in risk_cols]})

A tabela acima é o "dicionário" da RQ5: cada capacidade, sua gravidade (alta, média, baixa) e a classe de fraqueza CWE mais próxima. Os padrões exatos estão em `src/gitskills/patterns.py`.

## 4.1 Prevalência e exposição
**Objetivo:** saber quão comum é cada capacidade. `share_of_skills` conta cada texto uma vez; `share_of_occurrences` pondera pelo número de cópias, ou seja, quantas instalações carregam a capacidade (exposição).

In [ ]:
prev = pd.DataFrame({
    "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
    "share_of_skills": skills[risk_cols].mean().values,
    "share_of_occurrences": (skills[risk_cols].mul(skills["copies"], axis=0).sum() / skills["copies"].sum()).values,
}, index=[c[5:] for c in risk_cols]).sort_values(["severity", "share_of_skills"], ascending=[True, False])
save_table(prev.reset_index(names="capability"), "rq5_risk_prevalence")
high = [c for c in risk_cols if patterns.RISK_SEVERITY[c[5:]] == "high"]
print(f"skills with >=1 high-severity capability: {skills[high].any(axis=1).mean():.1%}")
prev

**O que os dados mostram:**
* **6,4%** das skills têm pelo menos uma capacidade de **gravidade alta**. A mais comum é a execução dinâmica (4,4%); `curl | sh` e desligar confirmações aparecem em 0,7% cada.
* As mais frequentes no geral são de gravidade média: mencionar segredos (**17,6%**) e instalar pacotes (**11,9%**).
* A exposição (por cópias) é um pouco **maior** que a prevalência em várias categorias (instalar pacotes: 11,9% → 15,0%): skills com essas capacidades tendem a ser mais copiadas.

**Cuidado: estes números são detecções, não instruções confirmadas.** A validação do notebook 05 (pré-rotulagem de 12 trechos por categoria, ainda sem o segundo avaliador) mostra precisões muito diferentes entre categorias:

| Capacidade | Detectado | Precisão | Prevalência ajustada |
|---|---|---|---|
| `curl \| sh` | 0,7% | 1,00 | **0,7%** |
| instalar pacotes | 11,9% | 0,83 | **~10%** |
| apagar arquivos | 1,1% | 0,83 | ~0,9% |
| privilégio | 2,1% | 0,75 | ~1,6% |
| segredos | 17,6% | 0,17 | ~3% |
| execução dinâmica | 4,4% | 0,08 | ~0,4% |
| desligar confirmações | 0,7% | 0,33 | ~0,2% |

Os detectores de `curl | sh`, instalação de pacotes, apagar arquivos e privilégio são confiáveis. Os de segredos, execução dinâmica e desligar confirmações disparam sobretudo em **menções** ("never store credentials", "never use `--no-verify`") ou em outro sentido da palavra ("eval" de *evaluation*). Com a precisão descontada, a fração de skills com alguma capacidade de gravidade alta cai de 6,4% para **no máximo ~2%**.

## 4.2 Deriva nas cópias editadas
**Objetivo:** a pergunta central. Em cada família, a referência é a variante mais copiada (proxy da original, que a base não identifica). Para cada outra variante, marcamos que capacidades ela **acrescentou** e quais **removeu**.

**Teste de McNemar:** se as edições fossem aleatórias em relação ao risco, acrescentar e remover seriam igualmente prováveis (50/50). Um p pequeno indica que um dos lados predomina.

In [ ]:
fam = reuse.name_families(skills)
drift = reuse.variant_drift(fam, risk_cols)
k, n = int(drift["any_added"].sum()), len(drift)
lo, hi = proportion_confint(k, n, method="wilson")
print(f"variants adding >=1 capability: {k:,}/{n:,} = {k/n:.1%} (95% CI {lo:.1%}-{hi:.1%})")
rows = []
for c in risk_cols:
    a, r = int(drift[f"added_{c}"].sum()), int(drift[f"removed_{c}"].sum())
    rows.append({"capability": c[5:], "severity": patterns.RISK_SEVERITY[c[5:]], "added": a, "removed": r,
                 "mcnemar_p": binomtest(a, a + r, 0.5).pvalue if a + r else np.nan})
mc = pd.DataFrame(rows)
save_table(mc, "rq5_mcnemar_added_vs_removed")
mc

**O que os dados mostram:**
* **17,3%** das 788.712 variantes (IC 95%: 17,2–17,3%) acrescentam pelo menos uma capacidade que a referência não tinha.
* Mas a deriva vai **nos dois sentidos**. Na maioria das categorias as edições **mais removem do que acrescentam** (instalar pacotes: 40.320 acréscimos × 51.713 remoções; segredos: 57.422 × 65.252; desligar confirmações: 4.477 × 5.527).
* **A exceção confiável é `curl | sh`:** baixar um script e executá-lo direto é acrescentado mais vezes do que removido (3.429 × 3.175, p = 0,002), e o detector teve precisão de 12/12 na validação.
* A execução dinâmica também aparece como "mais acrescentada" (19.016 × 15.605), mas **não deve ser interpretada**: o detector acerta só 1 em 12, porque a palavra "eval" quase sempre significa *evaluation* (avaliação de modelos). O efeito provavelmente reflete skills que passaram a falar de avaliação.
* Nas categorias de baixa precisão (segredos, desligar confirmações), parte das "remoções" pode ser só uma **regra** do tipo "nunca use `--no-verify`" que sumiu do texto.

Leitura: copiar e editar **não** torna as skills mais perigosas de forma geral; as cópias tendem a ser enxugadas. A forma mais direta de executar código remoto, `curl | sh`, é a que se acumula.

## 4.3 Que variantes acrescentam capacidades? (regressão logística)
**Objetivo:** ir além da média e ver **que características** se associam a uma variante acrescentar capacidade.

**Resposta (0/1):** a variante acrescenta pelo menos uma capacidade. **Preditores:** popularidade da referência (log de cópias), tamanho da família, se a variante é copiada entre donos, se traz scripts, se foi escrita com assistente.

**Como ler a razão de chances (odds ratio, OR):** OR = 1 não muda nada; OR = 1,46 aumenta a chance em 46%; OR = 0,69 reduz em 31%.

In [ ]:
d = drift.join(skills.set_index("file_sha")[["n_owners", "has_scripts", "first_commit_ai"]], on="file_sha")
d["family_size"] = d.groupby("family")["file_sha"].transform("size") + 1
d = d.assign(added=d["any_added"].astype(int), log_ref_copies=np.log1p(d["ref_copies"]),
             log_family_size=np.log(d["family_size"]), cross_owner=(d["n_owners"] > 1).astype(int),
             has_scripts=d["has_scripts"].fillna(0).astype(int), ai_authored=d["first_commit_ai"].notna().astype(int))
model = smf.logit("added ~ log_ref_copies + log_family_size + cross_owner + has_scripts + ai_authored", data=d).fit(disp=False)
odds = pd.DataFrame({"odds_ratio": np.exp(model.params), "ci_low": np.exp(model.conf_int()[0]),
                     "ci_high": np.exp(model.conf_int()[1]), "p": model.pvalues})
save_table(odds.reset_index(names="term"), "rq5_logit_odds_ratios")
odds.round(3)

**O que os dados mostram:**
* **Ter scripts** aumenta a chance de acrescentar capacidade em **46%** (OR 1,46).
* **Ser escrita com assistente** aumenta em **22%** (OR 1,22).
* **Famílias maiores** acrescentam mais (OR 1,36 por unidade de log).
* Variantes de skills **muito populares** acrescentam **menos** (OR 0,69): quem copia uma skill famosa tende a mantê-la.
* Ser copiada entre donos quase não importa (OR 0,97).

**Cuidado:** com 788 mil observações tudo é estatisticamente significativo, mas o modelo explica pouco (pseudo R² = 0,03). São **associações modestas**, não previsões. E a autoria por assistente só pode ser 1 onde há histórico (quase só em `.claude/skills/`), então parte do efeito pode ser o local.

## Conclusão da RQ5
* Detectamos alguma capacidade de gravidade alta em 6,4% das skills; descontada a precisão dos detectores, são **no máximo ~2%**. A capacidade confirmada mais comum é instalar pacotes (~10%).
* **17,3%** das variantes acrescentam alguma capacidade, mas as edições, no geral, **mais removem do que acrescentam**.
* A exceção confiável é **`curl | sh`**, que cresce nas cópias. A aparente alta da execução dinâmica é um artefato do detector ("eval" = avaliação).
* Variantes com scripts ou escritas com assistente acrescentam capacidades com mais frequência.

**Limitações:** (1) a precisão vem de uma pré-rotulagem de 108 trechos feita com apoio de um modelo de linguagem, ainda sem o segundo avaliador humano; (2) os detectores de segredos, execução dinâmica e desligar confirmações precisam ser refinados antes de os números dessas categorias irem para o artigo; (3) famílias agrupadas pelo nome juntam skills sem relação; (4) a referência é um proxy da original e, em muitas famílias, tem só 1 cópia ou empata com outras.

**Próximo passo:** o notebook 05 mede quanto confiar nos detectores.